# **Simulation Hedy 2025**

Team: TU Wien Space Team \
Project: Lamarr \
Rocket: Hedy \
Launched on 12.10.2025 at EuRoC


## Installs

In this section all needed libraries are installed and the needed classes imported

In [ ]:
%pip install rocketpy==1.12.1

In [ ]:
%pip install CoolProp

In [ ]:
from rocketpy import Environment, Rocket, Flight, Fluid, LiquidMotor, CylindricalTank, MassFlowRateBasedTank, TrapezoidalFins, FreeFormFins, RailButtons, NoseCone, Tail, Parachute, CompareFlights
import CoolProp.CoolProp as CP
import numpy as np

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parents[1]))   # walk up 1 level to root folder
from simulation.export_weather_data import export_weather_data
from simulation.custom_print_and_plot_functions import CustomPlots, CustomPrints

In [ ]:
# reload imported Python modules when you change their .py files
%load_ext autoreload
%autoreload 2

## Configuration

The length unit chosen here is millimeters to keep the values more readable. If necessary, values should be converted accordingly.

In [ ]:
# Rocket configuration data
env_config = {
    "date" : (2025, 10, 12, 17, 00, 00),  # yyyy, mm, dd, hh, mm, ss    # preset
    "latitude" : 39.39046096801758,       # Launch latitude             # preset
    "longitude" : -8.288534164428711,     # Launch longitude            # preset
    "max_height" : 9000,                  # m                           # preset    
    "timezone" : "Europe/Lisbon"          # GMT+1                       # preset
}
motor_config = {
    "holddown_time" : 1.4,                # s       # preset

    "nitrogen_tank" : {
        "length" : 235,                   # mm      # measured
        "CG_lox" : 2653,                  # mm      # measured # CG lox nitrogen tank
        "CG_ethanol" : 1450,              # mm      # measured # CG ethanol nitrogen tank
        "outer_diameter": 93,             # mm      # measured
        "volume" : 1.2,                   # l       # measured
        "massflow" : 0.03                 # kg/s    # estimated
    },
    "ethanol_tank" : {
        "length" : 573,                   # mm      # measured
        "CG" : 852,                       # mm      # measured
        "outer_diameter": 115,            # mm      # measured
        "volume" : 5.27,                  # l       # measured
        "massflow" : 0.45                 # kg/s    # tested
    },
    "lox_tank" : {
        "length" : 573,                   # mm      # measured
        "CG" : 1960,                      # mm      # measured
        "outer_diameter" : 115,           # mm      # measured
        "volume" : 5.27,                  # l       # measured
        "massflow" : 0.56                 # kg/s    # tested
    },
    "nozzle" : {
        "diameter" : 75,                  # mm      # measured
        "position" : -40                  # mm      # measured
    },

    "thrust": "thrust_euroc_launch.csv",  # N       # measured

    "temperature" : {
        "ethanol" : 298.15,               # K       # preset (= 25°C)
        "lox" : 93.15,                    # K       # preset (= -180°C)
        "nitrogen" : 298.15               # K       # preset (= 25°C)
    },
    "pressure" : {
        "ethanol" : 3000000,              # Pa      # preset
        "lox" : 3000000,                  # Pa      # preset
        "nitrogen" : 30000000             # Pa      # preset
    }
}

rocket_config = {
    "total_weight" : 17200,               # g       # weighed dry mass
    "total_length" : 3707,                # mm      # measured
    "total_CG" : 1813,                    # mm      # measured
    "moment_of_intertia_Z" : 0.0255,      # kg*m^2  # calculated
    "moment_of_intertia_XY" : 16.8,       # kg*m^2  # calculated
    "nosecone" : {
        "length" : 561,                   # mm      # measured    # 575 + 66 - 80 = 561
        "kind" : "lv haack"
    },
    "railbuttons" : {
        "upper" : 1646,                   # mm      # measured
        "lower" : 305                     # mm      # measured
    },
    "tailcone" : {
        "diameter" : 108,                 # mm      # measured
        "length" : 245,                   # mm      # measured
        "cylindrical_height" : 35         # mm      # measured
    },
    "rocket" : {
        "thickness" : 1.4,                # mm      # measured
        "diameter" : 132.8                # mm      # measured
    },
    "fins" : {
        "name" : "Biconvex-Freeform",
        "amount" : 4,
        "position" : 250,                 # mm      # measured
        "shape_points" : ((0,0),
                          (0.250,-0.012),
                          (0.250, 0.108),
                          (0.195, 0.108),
                          (0,0))          # m       # measured
    },
    "parachutes" : {
        "main" : {
            "cd_s" : 6.911503837897546,                     # calculated
            "trigger" : 450,              # m               # preset
            "sampling_rate" : 105,        # hz              # preset
            "lag" : 4,                    # s               # measured
            "noise" : (0, 8.3, 0.5)       # (pa, pa, pa)    # preset
        },
        "drogue" : {
            "cd_s" : 0.5336875,
            "trigger" : "apogee",         # m               # preset
            "sampling_rate" : 105,        # hz              # preset
            "lag" : 1,                    # s               # measured
            "noise" : (0, 8.3, 0.5)       # (pa, pa, pa)    # preset
        }
    }
}
flight_config = {
    "rail_length" : 11,                   # m       #Preset
    "inclination" : 84,                   # °       #Preset
    "heading" : 184,                      # °       #Preset
    "terminate_on_apogee" : False
}

## Environments Initialization




In this section the environments are initialized.
*   **envReanalysis**: environment with the weather data from the location and date of EuRoc'25


In [ ]:
#Ponte de Sor: 39.12368, -8.03333
#launch date: 12.10.2025, 17:00


latitude    = env_config["latitude"]
longitude   = env_config["longitude"]
timezone    = env_config["timezone"]
date        = env_config["date"]
max_height  = env_config["max_height"]

#Environment based on Forecast data for the EuRoC 2025
envReanalysis = Environment(max_expected_height = max_height)

envReanalysis.set_location(latitude = latitude, longitude = longitude)
envReanalysis.set_elevation("Open-Elevation")
envReanalysis.set_date(date, timezone = timezone)

envReanalysis.set_atmospheric_model(
    type="Reanalysis",
    file="euroc_weather.nc",
    dictionary="ECMWF"
)
envReanalysis.info()

## Simulation

### Tanks / Engine



In [ ]:
#holddown time
t_holddown          = motor_config["holddown_time"]

# tank height
h_nitrogen_tank     = motor_config["nitrogen_tank"]["length"]     / 1000
h_ethanol_tank      = motor_config["ethanol_tank"]["length"]      / 1000
h_lox_tank          = motor_config["lox_tank"]["length"]          / 1000
# OuterDiameter
OD_nitrogen_tank    = motor_config["nitrogen_tank"]["outer_diameter"]   / 1000
OD_ethanol_tank     = motor_config["ethanol_tank"]["outer_diameter"]    / 1000
OD_lox_tank         = motor_config["lox_tank"]["outer_diameter"]        / 1000
# volume
v_nitrogen_tank     = motor_config["nitrogen_tank"]["volume"]   / 1000
v_ethanol_tank      = motor_config["ethanol_tank"]["volume"]    / 1000
v_lox_tank          = motor_config["lox_tank"]["volume"]        / 1000
# massflows
mdot_nitrogen       = motor_config["nitrogen_tank"] ["massflow"]
mdot_ethanol        = motor_config["ethanol_tank"]["massflow"]
mdot_lox            = motor_config["lox_tank"]["massflow"]
# nozzle
nozzle_diameter     = motor_config["nozzle"]["diameter"]          / 1000

#thrust
thrust              = motor_config["thrust"]

# Temperature Lox & Ethanol
T_nitrogen          = motor_config["temperature"]["nitrogen"]
T_ethanol           = motor_config["temperature"]["ethanol"]
T_lox               = motor_config["temperature"]["lox"]
# pressure LOX & Ethanol
p_nitrogen          = motor_config["pressure"]["nitrogen"]
p_ethanol           = motor_config["pressure"]["ethanol"]
p_lox               = motor_config["pressure"]["lox"]


# Propellants

# define density
rho_nitrogen = CP.PropsSI("D","T",T_nitrogen,"P|gas",p_nitrogen,"N2")             # kg/m^3
rho_ethanol = CP.PropsSI("D", "T|liquid", T_ethanol, "P", p_ethanol, "ethanol")   # kg/m^3
rho_lox = CP.PropsSI("D", "T|liquid", T_lox, "P", p_lox, "oxygen")                # kg/m^3

# define fluids
nitrogen = Fluid(name = "N2", density = rho_nitrogen)
ethanol = Fluid(name = "ethanol", density = rho_ethanol)
lox = Fluid(name = "LOX", density = rho_lox)

# define tanks geometry
nitrogen_tank_shape = CylindricalTank(radius = OD_nitrogen_tank / 2, height = h_nitrogen_tank, spherical_caps = True)
ethanol_tank_shape = CylindricalTank(radius = OD_ethanol_tank / 2, height = h_ethanol_tank, spherical_caps = True)
lox_tank_shape = CylindricalTank(radius = OD_lox_tank / 2, height = h_lox_tank, spherical_caps = True)


m_nitrogen = v_nitrogen_tank * rho_nitrogen                   # m
m_ethanol = v_ethanol_tank * rho_ethanol                      # m
m_lox = v_lox_tank * rho_lox                                  # m

t_burn_nitrogen = (m_nitrogen / mdot_nitrogen)-0.001          # s


t_burn_ethanol = (m_ethanol / mdot_ethanol)
t_burn_lox = (m_lox / mdot_lox)

if(t_burn_ethanol<t_burn_lox):
  t_burn = t_burn_ethanol
  print(f"using ethanol burn time ({t_burn_ethanol}). LOX burntime is {t_burn_lox}")
else:
  t_burn = t_burn_lox
  print(f"using lox burn time ({t_burn_lox}). Ethanol burntime is {t_burn_ethanol}")


#account for holddown
t_burn -= t_holddown

m_lox = mdot_lox * t_burn + 0.0001              # adding a little more so python does not throw an error
m_ethanol = mdot_ethanol * t_burn + 0.0001
m_nitrogen = mdot_nitrogen * t_burn + 0.0001


# define tanks
nitrogen_tank = MassFlowRateBasedTank(
    name = "nitrogen tank",
    geometry = nitrogen_tank_shape,
    flux_time = t_burn,
    initial_liquid_mass = 0,                            # kg
    initial_gas_mass = m_nitrogen,
    liquid_mass_flow_rate_in = 0,                       # kg/s
    liquid_mass_flow_rate_out = 0,                      # kg/s
    gas_mass_flow_rate_in = 0,                          # kg/s
    gas_mass_flow_rate_out = lambda t: mdot_nitrogen,
    liquid = Fluid(name = "liquid", density = 0.0001),  # ignore
    gas = nitrogen
)
ethanol_tank = MassFlowRateBasedTank(
    name = "fuel tank",
    geometry = ethanol_tank_shape,
    flux_time = t_burn,
    initial_liquid_mass = m_ethanol,
    initial_gas_mass = 0,                               # kg
    liquid_mass_flow_rate_in = 0,                       # kg/s
    liquid_mass_flow_rate_out = lambda t: mdot_ethanol,
    gas_mass_flow_rate_in = lambda t: mdot_nitrogen,
    gas_mass_flow_rate_out = 0,                         # kg/s
    liquid = ethanol,
    gas = nitrogen
)

lox_tank = MassFlowRateBasedTank(
    name = "oxidizer tank",
    geometry = lox_tank_shape,
    flux_time = t_burn,
    initial_liquid_mass = m_lox,
    initial_gas_mass = 0,                               # kg
    liquid_mass_flow_rate_in = 0,                       # kg/s
    liquid_mass_flow_rate_out = lambda t: mdot_lox,
    gas_mass_flow_rate_in = lambda t: mdot_nitrogen,
    gas_mass_flow_rate_out = 0,                         # kg/s
    liquid = lox,
    gas = nitrogen
)

skuld = LiquidMotor(
    dry_mass = 0,                       # kg
    dry_inertia = (0,0,0),              # kg*m^2
    center_of_dry_mass_position = 0,    # m

    nozzle_radius = nozzle_diameter /2,
    nozzle_position = 0,                # m

    thrust_source = thrust,
    burn_time = t_burn,
    coordinate_system_orientation = "nozzle_to_combustion_chamber"
)

skuld.add_tank(tank = ethanol_tank, position    = motor_config["ethanol_tank"]["CG"]            / 1000)
skuld.add_tank(tank = lox_tank, position        = motor_config["lox_tank"]["CG"]                / 1000)
skuld.add_tank(tank = nitrogen_tank, position   = motor_config["nitrogen_tank"]["CG_ethanol"]   / 1000)
skuld.add_tank(tank = nitrogen_tank, position   = motor_config["nitrogen_tank"]["CG_lox"]       / 1000)

skuld.all_info()

### Rocket components

In [ ]:
#rocket
rocket_length     = rocket_config["total_length"]         / 1000
rocket_diameter   = rocket_config["rocket"]["diameter"]   / 1000
rocket_thickness  = rocket_config["rocket"]["thickness"]  / 1000


nosecone_length = (rocket_config["nosecone"]["length"]-80)    / 1000   # -80 mm to adjust for cylindrical section
nosecone_kind = rocket_config["nosecone"]["kind"]

nose_cone = NoseCone(
    length = nosecone_length,
    base_radius = rocket_diameter/2,
    kind = nosecone_kind
)

tailcone_cylindrical    = rocket_config["tailcone"]["cylindrical_height"] / 1000
tailcone_length         = rocket_config["tailcone"]["length"]             / 1000
tailcone_bottom_radius  = rocket_config["tailcone"]["diameter"]        /2 / 1000

tail = Tail(
    top_radius = rocket_diameter /2,
    bottom_radius = tailcone_bottom_radius,
    length = tailcone_length,
    rocket_radius = rocket_diameter/2
)

fin_amount        = rocket_config["fins"]["amount"]
fin_shape_points  = rocket_config["fins"]["shape_points"]
fin_name          = rocket_config["fins"]["name"]

fin_set = FreeFormFins(
   n = fin_amount,
   shape_points = fin_shape_points,
   rocket_radius=tailcone_bottom_radius,
   name = fin_name
)
fin_set.draw()

main_cd_s               = rocket_config["parachutes"]["drogue"]["cd_s"]                   # main didn't deploy fully, so adding another drogue as approximation
main_trigger            = rocket_config["parachutes"]["main"]["trigger"]
main_sampling_rate      = rocket_config["parachutes"]["main"]["sampling_rate"]
main_lag                = rocket_config["parachutes"]["main"]["lag"]
main_noise              = rocket_config["parachutes"]["main"]["noise"]

drogue_cd_s             = rocket_config["parachutes"]["drogue"]["cd_s"]
drogue_trigger          = rocket_config["parachutes"]["drogue"]["trigger"]
drogue_sampling_rate    = rocket_config["parachutes"]["drogue"]["sampling_rate"]
drogue_lag              = rocket_config["parachutes"]["drogue"]["lag"]
drogue_noise            = rocket_config["parachutes"]["drogue"]["noise"]

parachutes = {}

parachutes[0] = Parachute(
    name = "main",
    cd_s = main_cd_s,
    trigger = main_trigger,
    sampling_rate = main_sampling_rate,
    lag = main_lag,
    noise = main_noise
)

parachutes[1] = Parachute(
    name = "drogue",
    cd_s = drogue_cd_s,
    trigger = drogue_trigger,
    sampling_rate = drogue_sampling_rate,
    lag = drogue_lag,
    noise = drogue_noise
)


### Hedy

In [ ]:

total_mass        = rocket_config["total_weight"] / 1000

#inertia
inertia_x_y   = rocket_config["moment_of_intertia_XY"]
inertia_z     = rocket_config["moment_of_intertia_Z"]

upper_railbutton_position   = rocket_config["railbuttons"]["upper"] / 1000
lower_railbutton_position   = rocket_config["railbuttons"]["lower"] / 1000
fin_position                = rocket_config["fins"]["position"]     / 1000
nozzle_position             = motor_config["nozzle"]["position"]    / 1000

total_CG = rocket_config["total_CG"] / 1000

hedy = Rocket(
    radius = rocket_diameter /2,
    mass = total_mass,
    inertia = (inertia_x_y, inertia_x_y, inertia_z),
    power_off_drag = "./power_off_drag.csv",
    power_on_drag = "./power_on_drag.csv",
    center_of_mass_without_motor = total_CG,
    coordinate_system_orientation = "tail_to_nose"
)


hedy.add_motor(skuld, position = nozzle_position)


hedy.set_rail_buttons(upper_button_position= upper_railbutton_position, lower_button_position=lower_railbutton_position)

hedy.add_surfaces(surfaces=[nose_cone, fin_set, tail], positions=[rocket_length, fin_position, tailcone_length])

hedy.parachutes = list(parachutes.values())

hedy.all_info()

### Flight

In [ ]:
rail_length           = flight_config["rail_length"]
inclination           = flight_config["inclination"]
heading               = flight_config["heading"]
terminate_on_apogee   = flight_config["terminate_on_apogee"]

flight_reanalysis = Flight(
          rocket        = hedy,
          environment   = envReanalysis,
          rail_length   = rail_length,
          inclination   = inclination,
          heading       = heading,
          terminate_on_apogee = terminate_on_apogee,
          name          = "Reanalysis"
  )
print(f"Takeoff Mass: {hedy.total_mass(flight_reanalysis.out_of_rail_time)}")
flight_reanalysis.prints.out_of_rail_conditions()
flight_reanalysis.prints.apogee_conditions()
#flight_reanalysis.prints.impact_conditions()
flight_reanalysis.prints.maximum_values()
flight_reanalysis.plots.trajectory_3d()
flight_reanalysis.plots.stability_and_control_data()

#flight_reanalysis.prints.all()
#flight_reanalysis.plots.all()



#flight_reanalysis.all_info()

flight_reanalysis.export_kml()


## Comparsion

### Initialization

In [ ]:
from rocketpy.simulation.flight_data_importer import FlightDataImporter
from rocketpy import Function


def pressure_to_altitude(data):
    return 0.3048 * ((1 - (data / 1013.25) ** 0.190284) * 145366.45) # reference: https://www.weather.gov/media/epz/wxcalc/pressureAltitude.pdf


columns_map_cats_vega = {
    "ts": "time",
    "filteredAltitudeAGL": "altitude",
    "filteredAcceleration": "az",
    "latitude": "latitude",
    "longitude": "longitude",
    "Ax":"acceleration_x",
    "Ay":"acceleration_y",
    "Az":"acceleration_z",
    "Gx":"gyro_x",
    "Gy":"gyro_y",
    "Gz":"gyro_z",
    "velocity":"speed",
    "P": "pressure"

}
cats_vega_folder = "CATS_FLIGHT_DATA"
cats_vega_flight = FlightDataImporter(
    name="CATS Vega Flight Data",
    paths=[cats_vega_folder+"/filteredDataInfo.csv",cats_vega_folder+"/gnssInfo.csv",cats_vega_folder+"/flightInfo.csv",cats_vega_folder+"/imu.csv",cats_vega_folder+"/baro.csv"],
    columns_map=columns_map_cats_vega,
    units=None,
    interpolation="linear",
    extrapolation="zero",
    delimiter=",",
    encoding="utf-8",
)


columns_map_altimax = {
    "ZEIT":"time",
    "PRESS_FILTER":"pressure",
    "HEIGHT_FILTER":"altitude",
    "ACCEL":"acceleration",
    "SPEED":"speed"
}

altimax_flight = FlightDataImporter(
    name="Altimax Flight Data",
    paths=["ALTIMAX_FLIGHT_DATA/Hedy_Flug_EuRoC.csv"],
    columns_map=columns_map_altimax,
    units=None,
    interpolation="linear",
    extrapolation="zero",
    delimiter=",",
    encoding="utf-8",
)


columns_map_rcu = {
    "time":"time",
    "lora:gps_altitude:sensor":"altitude",
    "lora:gps_latitude:sensor":"latitude",
    "lora:gps_longitude:sensor":"longitude",
    "lora:gps_status:sensor":"gps_status",
    "lora:rcu_accel_x:sensor":"accel_x",
    "lora:rcu_accel_y:sensor":"accel_y",
    "lora:rcu_accel_z:sensor":"accel_z",
    "lora:rcu_gyro_x:sensor":"gyro_x",
    "lora:rcu_gyro_y:sensor":"gyro_y",
    "lora:rcu_gyro_z:sensor":"gyro_z",
    "lora:rcu_barometer:sensor":"pressure"
}

srad_flight = FlightDataImporter(
    name="SRAD Flight Data",
    paths=["RCU_FLIGHT_DATA/2025-10-12_EuRoC_launch_filtered.csv"],
    columns_map=columns_map_rcu,
    units=None,
    interpolation="linear",
    extrapolation="constant",
    delimiter=",",
    encoding="utf-8",
)

### Altitude

In [ ]:
data = srad_flight.pressure
altitude_srad = Function(data[np.isfinite(data[:,0]) & np.isfinite(data[:,1])])


apogee_srad = pressure_to_altitude(altitude_srad.min)

apogee_actual = (apogee_srad + cats_vega_flight.altitude.max) /2
apogee_simulated = flight_reanalysis.apogee - flight_reanalysis.env.elevation
apogee_error = abs(apogee_actual - apogee_simulated)
apogee_percentage_error = apogee_error / apogee_actual * 100

print(f"Actual apogee: {apogee_actual:.2f} m (AGL)")
print(f"Simulated apogee: {apogee_simulated:.2f} m (AGL)")
print(f"Error: {apogee_error:.2f} m")
print(f"Percentage Error: {apogee_percentage_error:.2f}%")


Function.compare_plots(
    [
        (flight_reanalysis.altitude, "RocketPy"),
        (cats_vega_flight.altitude, "CATS Vega"),
        (altimax_flight.altitude, "Altimax"),
        (pressure_to_altitude(altitude_srad), "SRAD")
    ],
    title="Altitude Comparison",
    xlabel="Time (s)",
    ylabel="Altitude (m)",
)

### GNSS

In [ ]:
Function.compare_plots(
    [
        (flight_reanalysis.latitude, "RocketPy"),
        (srad_flight.latitude, "SRAD"),
        (cats_vega_flight.latitude, "CATS")
    ],
    title="Latitude Comparison",
    xlabel="Time (s)",
    ylabel="Latitude (deg)",
)
Function.compare_plots(
    [
        (flight_reanalysis.longitude, "RocketPy"),
        (srad_flight.longitude, "SRAD"),
        (cats_vega_flight.longitude, "CATS")
    ],
    title="Longitude Comparison",
    xlabel="Time (s)",
    ylabel="Longitude (deg)",
)

### Pressure

In [ ]:
pressure_actual = altimax_flight.pressure.min
pressure_simulated = flight_reanalysis.pressure.min
pressure_error = abs(pressure_actual - pressure_simulated)
pressure_percentage_error = pressure_error / pressure_actual * 100

print(f"Actual min pressure: {pressure_actual:.2f} Pa")
print(f"Simulated min pressure: {pressure_simulated:.2f} Pa")
print(f"Error: {pressure_error:.2f} Pa")
print(f"Percentage Error: {pressure_percentage_error:.2f}%")


Function.compare_plots(
    [
        (flight_reanalysis.pressure, "RocketPy"),
        (cats_vega_flight.pressure, "CATS vega"),
        (altimax_flight.pressure, "Altimax"),
        (srad_flight.pressure*100, "SRAD")
    ],
    title="Pressure Comparison",
    xlabel="Time (s)",
    ylabel="Pressure (Pa)",
)

### Acceleration

In [ ]:
acceleration_actual = altimax_flight.acceleration.crop([(0, 20)]).max
acceleration_simulated = flight_reanalysis.acceleration.crop([(0, 20)]).max
acceleration_error = abs(acceleration_actual - acceleration_simulated)
acceleration_percentage_error = acceleration_error / acceleration_actual * 100

print(f"Actual max acceleration during burn: {acceleration_actual:.2f} m/s2")
print(f"Simulated max acceleration during burn: {acceleration_simulated:.2f} m/s2")
print(f"Error: {acceleration_error:.2f} m/s2")
print(f"Percentage Error: {acceleration_percentage_error:.2f}%")

Function.compare_plots(
    [
        (flight_reanalysis.ay.crop([(0, 100)]), "RocketPy"),
        (cats_vega_flight.az.crop([(0, 100)]) /10 *(-1), "CATS Vega"),
        (altimax_flight.acceleration.crop([(0, 100)]) /10 *(-1), "Altimax"),
        (srad_flight.accel_y.crop([(0, 100)]), "SRAD Acceleration Y")
    ],
    title="Acceleration Comparison",
    xlabel="Time (s)",
    ylabel="Vertical acceleration (m/s^2)",
)

### Speed

In [ ]:
speed_actual = altimax_flight.speed.max
speed_simulated = flight_reanalysis.speed.max
speed_error = abs(speed_actual - speed_simulated)
speed_percentage_error = speed_error /speed_actual * 100

print(f"Actual max speed: {speed_actual:.2f} m/s2")
print(f"Simulated max speed: {speed_simulated:.2f} m/s2")
print(f"Error: {speed_error:.2f} m/s2")
print(f"Percentage Error: {speed_percentage_error:.2f}%")

Function.compare_plots(
    [
        (flight_reanalysis.vz.crop([(0, 100)]), "RocketPy"),
        (cats_vega_flight.speed.crop([(0, 100)]), "CATS Vega"),
        (altimax_flight.speed.crop([(0, 100)]), "Altimax")
    ],
    title="Speed Comparison",
    xlabel="Time (s)",
    ylabel="Speed (m/s)",
)

flights_for_comparison = [("RocketPy", flight_reanalysis), ("CATS VEGA", cats_vega_flight), ("ALTIMAX", altimax_flight)]
custom_plots = CustomPlots(
    flight_forecast=[f for _, f in flights_for_comparison],
    motor=[skuld] * len(flights_for_comparison),
    plot_title=[name for name, _ in flights_for_comparison],
    rocket=[hedy] * len(flights_for_comparison),
    rocket_config=[rocket_config] * len(flights_for_comparison),
)

# custom_plots.plot_stability_and_cg_cp_position()
# flight_forecast.plots.stability_and_control_data()
# custom_plots.plot_angle_of_attack_and_attitude_angle()
custom_plots.plot_motion_over_time(time_interval=(0, 250))
# flight_forecast.plots.trajectory_3d()